# Water Quality Prediction — Milestone 1: Setup & Dataset Loading

**Dataset:** `water_potability.csv` — 3,276 water samples, each labelled 0 (unsafe) or 1 (safe)  
**Goal of this notebook:** confirm the environment works, load the dataset, understand its shape and quirks before we touch anything.

> Think of this notebook like the first commit on a new project — get the scaffolding right before writing any real logic.

## Cell 1 — Imports

Pulling in everything we'll need across all milestones. Matplotlib gets a non-interactive backend so figures render cleanly in notebooks.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # headless rendering — figures saved to disk, not popped up
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import joblib
import os

print('pandas', pd.__version__)
print('numpy', np.__version__)
print('sklearn and xgboost loaded')
print('\n✅ T1.1 PASS — all imports OK')

pandas 2.3.3
numpy 2.0.2
sklearn and xgboost loaded

✅ T1.1 PASS — all imports OK


## Cell 2 — Load Dataset

Reading from the `data/` folder. The CSV has 9 numeric feature columns and one binary target (`Potability`).  
We'll name the DataFrame `water_df` throughout — more specific than a plain `df` and makes the domain obvious when you're deep in preprocessing cells.

In [2]:
water_df = pd.read_csv('../data/water_potability.csv')

print(f'Shape: {water_df.shape}')
print(f'Columns: {list(water_df.columns)}')
print()
water_df.head()

Shape: (3276, 10)
Columns: ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity', 'Potability']



,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
0,NaN,204.890455,20791.318981,7.300212,368.516441,564.308654,10.379783,86.990970,2.963135,0
1,3.716080,129.422921,18630.057858,6.635246,NaN,592.885359,15.180013,56.329076,4.500656,0
2,8.099124,224.236259,19909.541732,9.275884,NaN,418.606213,16.868637,66.420093,3.055934,0
3,8.316766,214.373394,22018.417441,8.059332,356.886136,363.266516,18.436524,100.341674,4.628771,0
4,9.092223,181.101509,17978.986339,6.546600,310.135738,398.410813,11.558279,31.997993,4.075075,0


## Cell 3 — Basic Dataset Statistics

`.describe()` is the equivalent of a quick `SELECT MIN, MAX, AVG, STDDEV` on every column.  
Notice that `ph`, `Sulfate`, and `Trihalomethanes` will show a `count` below 3276 — that's where the nulls are hiding.

In [3]:
print('=== Descriptive Statistics ===')
display(water_df.describe().round(3))

print('\n=== Data Types ===')
print(water_df.dtypes)

print('\n=== Missing Values per Column ===')
missing = water_df.isnull().sum()
print(missing[missing > 0])  # only show columns that actually have nulls

total_missing = missing.sum()
pct_missing = (total_missing / water_df.size) * 100
print(f'\nTotal missing cells: {total_missing} ({pct_missing:.1f}% of dataset)')

=== Descriptive Statistics ===


,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity,Potability
count,2785.000,3276.000,3276.000,3276.000,2495.000,3276.000,3276.000,3114.000,3276.000,3276.000
mean,7.081,196.369,22014.093,7.122,333.776,426.205,14.285,66.396,3.967,0.390
std,1.594,32.880,8768.571,1.583,41.417,80.824,3.308,16.175,0.780,0.488
min,0.000,47.432,320.943,0.352,129.000,181.484,2.200,0.738,1.450,0.000
25%,6.093,176.851,15666.690,6.127,307.699,365.734,12.066,55.845,3.440,0.000
50%,7.037,196.968,20927.834,7.130,333.074,421.885,14.218,66.622,3.955,0.000
75%,8.062,216.667,27332.762,8.115,359.950,481.792,16.558,77.337,4.500,1.000
max,14.000,323.124,61227.196,13.127,481.031,753.343,28.300,124.000,6.739,1.000



=== Data Types ===
ph                 float64
Hardness           float64
Solids             float64
Chloramines        float64
Sulfate            float64
Conductivity       float64
Organic_carbon     float64
Trihalomethanes    float64
Turbidity          float64
Potability           int64
dtype: object

=== Missing Values per Column ===
ph                 491
Sulfate            781
Trihalomethanes    162
dtype: int64

Total missing cells: 1434 (4.4% of dataset)


## Cell 4 — Class Distribution

The target is binary — 0 means unsafe, 1 means safe.  
This dataset is not balanced: ~61% unsafe, ~39% safe.  
That matters later: a model that just always predicts 'unsafe' gets 61% accuracy without learning anything. This is why we'll use F1-score and AUC alongside accuracy when evaluating models.

In [4]:
class_counts = water_df['Potability'].value_counts()
safe_pct = class_counts[1] / len(water_df) * 100
unsafe_pct = class_counts[0] / len(water_df) * 100

print('Class Distribution:')
print(f'  Not Safe (0): {class_counts[0]} samples  ({unsafe_pct:.1f}%)')
print(f'  Safe     (1): {class_counts[1]} samples  ({safe_pct:.1f}%)')
print()
print('⚠️  Imbalance noted — accuracy alone will be misleading. F1 and AUC are the real metrics.')

Class Distribution:
  Not Safe (0): 1998 samples  (61.0%)
  Safe     (1): 1278 samples  (39.0%)

⚠️  Imbalance noted — accuracy alone will be misleading. F1 and AUC are the real metrics.


## Cell 5 — Column Overview

Quick check on value ranges for each feature. Useful sanity check — if pH goes above 14 or below 0 those are measurement errors, not real data.

In [5]:
feature_cols = [c for c in water_df.columns if c != 'Potability']

header = '{:<22} {:>10} {:>10} {:>10} {:>8}'.format('Feature', 'Min', 'Max', 'Mean', 'Nulls')
print(header)
print('-' * 65)
for col in feature_cols:
    nulls = water_df[col].isnull().sum()
    null_marker = '{} (missing)'.format(nulls) if nulls > 0 else str(nulls)
    row = '{:<22} {:>10.2f} {:>10.2f} {:>10.2f} {:>14}'.format(
        col, water_df[col].min(), water_df[col].max(), water_df[col].mean(), null_marker
    )
    print(row)

Feature                       Min        Max       Mean    Nulls
-----------------------------------------------------------------
ph                           0.00      14.00       7.08  491 (missing)
Hardness                    47.43     323.12     196.37              0
Solids                     320.94   61227.20   22014.09              0
Chloramines                  0.35      13.13       7.12              0
Sulfate                    129.00     481.03     333.78  781 (missing)
Conductivity               181.48     753.34     426.21              0
Organic_carbon               2.20      28.30      14.28              0
Trihalomethanes              0.74     124.00      66.40  162 (missing)
Turbidity                    1.45       6.74       3.97              0


---

## Milestone 1 Tests

Run these cells to confirm M1 is complete. A clean run with no `AssertionError` = milestone passed.

> Same discipline as not merging a PR until CI is green.

In [6]:
# T1.1 — already ran at the top of this notebook (import cell)
# Re-asserting here so all tests are in one place

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import joblib

print('✅ T1.1 PASS — all imports OK')

✅ T1.1 PASS — all imports OK


In [7]:
# T1.2 — dataset loads with correct shape
water_df = pd.read_csv('../data/water_potability.csv')

assert water_df.shape[1] == 10, f"Expected 10 columns, got {water_df.shape[1]}"
assert water_df.shape[0] > 3000, f"Expected 3000+ rows, got {water_df.shape[0]}"
assert 'Potability' in water_df.columns, "Target column 'Potability' is missing"

print(f'✅ T1.2 PASS — dataset shape: {water_df.shape}')

✅ T1.2 PASS — dataset shape: (3276, 10)


In [8]:
# T1.3 — target column is strictly binary
unique_vals = set(water_df['Potability'].unique())

assert unique_vals == {0, 1}, f"Expected only 0 and 1, got: {unique_vals}"

print(f'✅ T1.3 PASS — Potability values: {unique_vals}')

✅ T1.3 PASS — Potability values: {np.int64(0), np.int64(1)}


In [9]:
# T1.4 — folder structure is in place
import os

# paths are relative to the project root, not notebooks/
required_dirs = ['../data', '../notebooks', '../models', '../reports/figures', '../docs']
for folder in required_dirs:
    assert os.path.exists(folder), f"Missing folder: {folder}"

print('✅ T1.4 PASS — folder structure is in place')

✅ T1.4 PASS — folder structure is in place


In [10]:
# Final summary — what we know heading into M2
print('=== Milestone 1 Complete ===')
print(f'Dataset shape:   {water_df.shape}')
print(f'Features:        {[c for c in water_df.columns if c != "Potability"]}')
print(f'Target:          Potability (0=unsafe, 1=safe)')
print(f'Missing values:  ph={water_df["ph"].isnull().sum()}, Sulfate={water_df["Sulfate"].isnull().sum()}, Trihalomethanes={water_df["Trihalomethanes"].isnull().sum()}')
print(f'Class split:     {water_df["Potability"].value_counts()[0]} unsafe / {water_df["Potability"].value_counts()[1]} safe')
print()
print('Next: Milestone 2 — EDA (distributions, heatmap, boxplots, class imbalance)')

=== Milestone 1 Complete ===
Dataset shape:   (3276, 10)
Features:        ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity']
Target:          Potability (0=unsafe, 1=safe)
Missing values:  ph=491, Sulfate=781, Trihalomethanes=162
Class split:     1998 unsafe / 1278 safe

Next: Milestone 2 — EDA (distributions, heatmap, boxplots, class imbalance)
